## Import Libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DecimalType
from pyspark.sql.functions import col, lower, upper, when, trim, count, lit

## Reading from bronze table

In [0]:
df = spark.table("olist.bronze.order_payments")
df.display()

## Overview of the table

In [0]:
# Table info
print("=== Schema ===")
df.printSchema()

print("=== Row Count ===")
print(f"Total rows: {df.count()}")

print("=== Null Counts per Column ===")
df.select([
    F.count(F.when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).display()

In [0]:
# Group the DataFrame by 'payment_type' and display the count of records for each payment type
display(df.groupBy("payment_type").count())



In [0]:
# Display distinct values of 'payment_installments' column
df.select("payment_installments").distinct().display()

In [0]:
# Display distinct values of 'payment_sequential' column
df.select("payment_sequential").distinct().display()

## Transformations

### 1. Trim all whitespaces from string columns

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
    

### 2. Normalize improperly represented nulls in string columns

In [0]:
NULL_STRINGS = ["", "null", "none", "n/a", "na", "unknown", "-", "not_defined"]

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(
            field.name,
            when(lower(trim(col(field.name))).isin(NULL_STRINGS), None)
            .otherwise(col(field.name)))

### 3. Normalize payment_type to lowercase

In [0]:
df = df.withColumn("payment_type", lower(col("payment_type")))

### 4. Cast monetary column to DECIMAL(10,2)

In [0]:
# Keep cent-level precision for accurate gold-layer revenue calculations.
# DECIMAL(10,2) is the standard type for currency in data warehousing.
df = df.withColumn("payment_value", col("payment_value").cast(DecimalType(10, 2)))

### 5. Validate numeric values

In [0]:
# payment_value should be > 0 (a zero-value payment doesn't represent a real transaction)
# payment_installments should be >= 1 (at least one installment for any valid payment)
# payment_sequential should be >= 1
df = df.withColumn(
    "has_valid_payment",
    (col("payment_value") > 0) &
    (col("payment_installments") >= 1) &
    (col("payment_sequential") >= 1)
)

### 6. Derive useful analytical columns

In [0]:
# installment_value: value per installment — useful for affordability / credit analysis
# is_installment_payment: flag for payments split across multiple installments
# is_voucher: quick flag for voucher-based payments (often combined with other methods)
df = df.withColumn(
    "installment_value",
    when(col("payment_installments") > 0,
         (col("payment_value") / col("payment_installments")).cast(DecimalType(10, 2)))
    .otherwise(None)
).withColumn(
    "is_installment_payment",
    col("payment_installments") > 1
).withColumn(
    "is_voucher",
    col("payment_type") == lit("voucher")
)

### 7. Handle Nulls

In [0]:
# Drop rows missing any component of the composite primary key
df = df.filter(
    col("order_id").isNotNull() &
    col("payment_sequential").isNotNull()
)

### 8. Remove duplicates

In [0]:
# Check for duplicates before removing
print(f"Rows before deduplication: {df.count()}")
print(f"Distinct rows based on primary key: {df.select('order_id', 'payment_sequential').distinct().count()}")

# Drop duplicates based on composite primary key
df = df.dropDuplicates(['order_id', 'payment_sequential'])

print(f"Rows after deduplication: {df.count()}")

## Quality Checks

In [0]:
print(f"Total rows after cleaning: {df.count()}")
print(f"Unique (order_id, payment_sequential): {df.select('order_id', 'payment_sequential').distinct().count()}")
print(f"Distinct orders: {df.select('order_id').distinct().count()}")
print(f"Rows with invalid payments: {df.filter(~col('has_valid_payment')).count()}")
print(f"Null payment_type: {df.filter(col('payment_type').isNull()).count()}")
print(f"Null payment_value: {df.filter(col('payment_value').isNull()).count()}")
print(f"Null payment_installments: {df.filter(col('payment_installments').isNull()).count()}")

print("\n=== Payment type distribution ===")
df.groupBy("payment_type").count().orderBy(F.desc("count")).display()

print("\n=== Installment distribution ===")
df.groupBy("payment_installments").count().orderBy("payment_installments").display()

df.display()

## Write to silver layer

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("olist.silver.order_payments")